# Stress Score Prediction V5 — Median ExtraTrees

V4의 누수 없는 전처리와 행 단위 파생변수를 유지하고, MAE에 맞춰 각 트리 예측의 평균 대신 중앙값을 사용하는 버전입니다.


## 0. Import & Data Load


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score


In [2]:
DATA_DIR = Path('/Users/bitsaem/Desktop/stress_project/open')

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)


Train shape: (3000, 18)
Test shape: (3000, 17)


## 1. Data Split


In [3]:
TARGET = 'stress_score'
ID_COL = 'ID'

X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET].copy()

X_test = test.drop(columns=[ID_COL])
test_ids = test[ID_COL].copy()


## 2. Row-level Feature Engineering


In [4]:
def add_features(df):
    """다른 행이나 데이터셋 통계를 사용하지 않는 행 단위 파생변수 생성 함수."""
    df = df.copy()

    height_m = df['height'] / 100
    sbp = df['systolic_blood_pressure']
    dbp = df['diastolic_blood_pressure']

    df['bmi'] = df['weight'] / (height_m ** 2)
    df['pulse_pressure'] = sbp - dbp
    df['mean_arterial_pressure'] = (sbp + 2 * dbp) / 3
    df['blood_pressure_ratio'] = sbp / (dbp + 1e-6)
    df['cholesterol_glucose_ratio'] = (
        df['cholesterol'] / (df['glucose'] + 1e-6)
    )
    df['missing_count'] = df.isna().sum(axis=1)

    return df


X_v5 = add_features(X)
X_test_v5 = add_features(X_test)

print('Original shape:', X.shape)
print('V5 shape:', X_v5.shape)


Original shape: (3000, 16)
V5 shape: (3000, 22)


## 3. Leakage-safe Preprocessing


In [5]:
categorical_cols = X_v5.select_dtypes(
    include=['object', 'category']
).columns.tolist()

numerical_cols = X_v5.select_dtypes(
    include=['number']
).columns.tolist()

print('Categorical:', categorical_cols)
print('Numerical:', numerical_cols)


Categorical: ['gender', 'activity', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern', 'edu_level']
Numerical: ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure', 'diastolic_blood_pressure', 'glucose', 'bone_density', 'mean_working', 'bmi', 'pulse_pressure', 'mean_arterial_pressure', 'blood_pressure_ratio', 'cholesterol_glucose_ratio', 'missing_count']


In [6]:
categorical_pipeline = Pipeline([
    (
        'imputer',
        SimpleImputer(
            strategy='constant',
            fill_value='missing'
        )
    ),
    (
        'encoder',
        OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=False
        )
    )
])

numerical_pipeline = Pipeline([
    (
        'imputer',
        SimpleImputer(
            strategy='median',
            add_indicator=True
        )
    )
])

preprocessor_v5 = ColumnTransformer([
    ('categorical', categorical_pipeline, categorical_cols),
    ('numerical', numerical_pipeline, numerical_cols)
])


## 4. Median ExtraTrees & Cross-validation


In [7]:
model_v5 = ExtraTreesRegressor(
    n_estimators=600,
    min_samples_leaf=1,
    max_features=1,
    random_state=42,
    n_jobs=-1
)

pipeline_v5 = Pipeline([
    ('preprocessor', preprocessor_v5),
    ('model', model_v5)
])


In [8]:
def median_forest_predict(fitted_pipeline, X_data):
    """전처리 후 개별 트리 예측의 행별 중앙값을 반환합니다."""
    transformed = fitted_pipeline.named_steps['preprocessor'].transform(X_data)
    forest = fitted_pipeline.named_steps['model']

    tree_predictions = np.column_stack([
        tree.predict(transformed)
        for tree in forest.estimators_
    ])

    return np.median(tree_predictions, axis=1)


cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

v5_fold_mae = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X_v5), start=1):
    fold_pipeline = clone(pipeline_v5)

    fold_pipeline.fit(
        X_v5.iloc[train_idx],
        y.iloc[train_idx]
    )

    valid_pred = median_forest_predict(
        fold_pipeline,
        X_v5.iloc[valid_idx]
    )

    fold_mae = np.mean(
        np.abs(y.iloc[valid_idx].to_numpy() - valid_pred)
    )
    v5_fold_mae.append(fold_mae)

    print(f'Fold {fold} MAE: {fold_mae:.6f}')

v5_fold_mae = np.array(v5_fold_mae)

V3_CV_MAE = 0.174747
V4_CV_MAE = 0.170323

print('V5 Fold MAE:', np.round(v5_fold_mae, 6))
print('V3 Mean MAE:', V3_CV_MAE)
print('V4 Mean MAE:', V4_CV_MAE)
print('V5 Mean MAE:', round(v5_fold_mae.mean(), 6))
print('V5 vs V4:', round(v5_fold_mae.mean() - V4_CV_MAE, 6))


Fold 1 MAE: 0.137575
Fold 2 MAE: 0.152475
Fold 3 MAE: 0.151167
Fold 4 MAE: 0.161617
Fold 5 MAE: 0.152842
V5 Fold MAE: [0.137575 0.152475 0.151167 0.161617 0.152842]
V3 Mean MAE: 0.174747
V4 Mean MAE: 0.170323
V5 Mean MAE: 0.151135
V5 vs V4: -0.019188


## 5. Final Training & Prediction


In [9]:
pipeline_v5.fit(X_v5, y)

pred_v5 = median_forest_predict(
    pipeline_v5,
    X_test_v5
)
pred_v5 = np.clip(pred_v5, 0, 1)

print('Prediction shape:', pred_v5.shape)
print('Prediction range:', pred_v5.min(), pred_v5.max())


Prediction shape: (3000,)
Prediction range: 0.0 1.0


## 6. Submission


In [10]:
submission_v5 = submission.copy()
submission_v5[ID_COL] = test_ids
submission_v5[TARGET] = pred_v5

output_path_v5 = DATA_DIR.parent / 'submit_v5_median_extratrees.csv'
submission_v5.to_csv(output_path_v5, index=False)

print('Saved:', output_path_v5)
submission_v5.head()


Saved: /Users/bitsaem/Desktop/stress_project/submit_v5_median_extratrees.csv


,ID,stress_score
0,TEST_0000,0.62
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.41
4,TEST_0004,0.53
